# 05 — Polymarket evaluation

Notebook 04 evaluated the model against **BestFightOdds-derived no-vig prices** with a Kalshi-like 7% fee assumption. This notebook re-runs the same evaluation against **actual Polymarket closing prices** scraped from the Polymarket API.

Why this matters: Polymarket is the actual market we'd deploy on. BestFightOdds-Kalshi-like was a theoretical scenario derived by stripping the vig from aggregated sportsbook quotes. Real prediction markets clear at different prices, and the model's edge against real Polymarket prices is what determines real-world deployment ROI.

## Data

- **Polymarket historical**: 936 fights between 2024-04-13 and 2026-05-16 with closing prices (probability format) and winners.
- **Matching**: 585 of 931 (63%) matched to Kaggle by date + fighter name (with fuzzy last-name matching for early-2024 rows that used last names only).
- All matched fights sit within the test window (2024-04-13 → 2026-03-28). Post-test Polymarket fights (90+ from 2026-04-04 onwards) have no features in our Kaggle data yet.

## Key features of Polymarket pricing

- Closing prices sum to exactly 1.0 (binary contract on each side) — these are **no-vig market consensus probabilities by construction**.
- Decimal odds: `dec_a = 1 / closing_price_a` (and same for b).
- Real Polymarket execution has a small per-trade fee (~2% on winnings typical) and bid-ask spread. We evaluate three fee scenarios: 0%, 2%, 7%.

## Model

`v3_catboost_full2000_trainval` — the deployed champion. Trained on train+val (2010-2023); 2024+ is held out from training. This evaluation tells us: given the same predictions we ran against BestFightOdds in notebook 04, what does the realized ROI look like priced against Polymarket?

In [ ]:
import sys, unicodedata
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ufc_pred.backtest.bet_eval import evaluate_bets, evaluate_bets_kelly, american_to_decimal
from ufc_pred.backtest.metrics import evaluate, american_to_implied_prob
from ufc_pred.features.skill_v3_pipeline import OUTPUT as SKILL_V3_PARQUET
from ufc_pred.features.static_v1 import prepare
from ufc_pred.ingest.kaggle_mdabbert import HISTORY_PARQUET

pd.set_option('display.precision', 3)

## Load Polymarket + Kaggle, match by date and name

In [ ]:
import re
APOSTROPHES = "\'’ʼ`‘"

def norm(s):
    """Plain ASCII-fold + lowercase (used to display friendly results)."""
    if pd.isna(s): return ""
    return unicodedata.normalize("NFKD", str(s)).encode("ascii","ignore").decode("ascii").lower().strip()

def deep_norm(s):
    """Stricter normalize: also strip apostrophes/hyphens/periods/commas."""
    if pd.isna(s) or s is None: return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii","ignore").decode("ascii")
    for ch in APOSTROPHES + "-.,":
        s = s.replace(ch, "")
    return re.sub(r"\s+", " ", s).strip().lower()

def last_token(s):
    s = deep_norm(s)
    return s.split()[-1] if s else ""

# Polymarket
poly = pd.read_parquet(ROOT / 'data/raw/polymarket/historical_2024-04-13_to_2026-05-24.parquet')
poly = poly.dropna(subset=['closing_price_a','closing_price_b','winner']).copy()
poly['fd_n']  = pd.to_datetime(poly['fight_date']).dt.tz_localize(None).dt.normalize()
poly['fd_m1'] = poly['fd_n'] - pd.Timedelta(days=1)
poly['a_dn']  = poly['fighter_a'].map(deep_norm)
poly['b_dn']  = poly['fighter_b'].map(deep_norm)
poly['a_last'] = poly['fighter_a'].map(last_token)
poly['b_last'] = poly['fighter_b'].map(last_token)

# Kaggle
fights = pd.read_parquet(HISTORY_PARQUET)
fights = fights[fights['Winner'].isin(['Red','Blue'])].copy()
fights['date'] = pd.to_datetime(fights['date'])
fights['R_dn']  = fights['R_fighter'].map(deep_norm)
fights['B_dn']  = fights['B_fighter'].map(deep_norm)
fights['R_last'] = fights['R_fighter'].map(last_token)
fights['B_last'] = fights['B_fighter'].map(last_token)

sk = pd.read_parquet(SKILL_V3_PARQUET)
sk['date'] = pd.to_datetime(sk['date'])
fights = fights.merge(
    sk[['date','R_fighter','B_fighter','skill_diff_mean','skill_diff_std']],
    on=['date','R_fighter','B_fighter'], how='left', validate='many_to_one',
)
print(f'Polymarket rows: {len(poly)}')
print(f'Kaggle rows:     {len(fights)}')


In [ ]:
from rapidfuzz import fuzz

fights_by_date = {d: g for d, g in fights.groupby('date')}

def candidates_local(row):
    """Pool of Kaggle fights within ±2 days of either Polymarket date convention."""
    dates = set()
    for d in (row['fd_n'], row['fd_m1']):
        for off in (-1, 0, 1):
            dates.add(d + pd.Timedelta(days=off))
    pools = [fights_by_date[d] for d in dates if d in fights_by_date]
    return pd.concat(pools) if pools else pd.DataFrame()

def match_one(row):
    pool = candidates_local(row)
    if len(pool) > 0:
        a, b = row['a_dn'], row['b_dn']
        a_l, b_l = row['a_last'], row['b_last']
        # exact full
        h = pool[((pool['R_dn']==a) & (pool['B_dn']==b)) | ((pool['R_dn']==b) & (pool['B_dn']==a))]
        if len(h) == 1:
            k = h.iloc[0]; return k, k['R_dn']==a, 'exact_full'
        # exact last
        h = pool[((pool['R_last']==a_l) & (pool['B_last']==b_l)) | ((pool['R_last']==b_l) & (pool['B_last']==a_l))]
        if len(h) == 1:
            k = h.iloc[0]; return k, k['R_last']==a_l, 'exact_last'
        # substring
        h = pool[((pool['R_dn'].str.contains(a_l, regex=False)) & (pool['B_dn'].str.contains(b_l, regex=False))) |
                 ((pool['R_dn'].str.contains(b_l, regex=False)) & (pool['B_dn'].str.contains(a_l, regex=False)))]
        if len(h) == 1:
            k = h.iloc[0]; return k, a_l in k['R_dn'], 'substr'
        # fuzzy last name (ratio ≥ 88)
        def fp(k):
            return (max(fuzz.ratio(a_l, k['R_last']), fuzz.ratio(a_l, k['B_last'])) +
                    max(fuzz.ratio(b_l, k['R_last']), fuzz.ratio(b_l, k['B_last']))) / 2
        pool = pool.copy(); pool['fuz'] = pool.apply(fp, axis=1)
        best = pool.sort_values('fuz', ascending=False).head(2)
        if len(best) > 0 and best.iloc[0]['fuz'] >= 88 and (len(best) == 1 or best.iloc[0]['fuz'] - best.iloc[1]['fuz'] >= 5):
            k = best.iloc[0]
            return k, fuzz.ratio(a_l, k['R_last']) >= fuzz.ratio(a_l, k['B_last']), 'fuzzy'
    # Wide ±14 day window for rescheduled fights
    d = row['fd_n']
    w = fights[(fights['date'] >= d - pd.Timedelta(days=14)) & (fights['date'] <= d + pd.Timedelta(days=14))]
    h = w[((w['R_last']==row['a_last']) & (w['B_last']==row['b_last'])) | ((w['R_last']==row['b_last']) & (w['B_last']==row['a_last']))]
    if len(h) == 1:
        k = h.iloc[0]; return k, k['R_last']==row['a_last'], 'wide_14d'
    return None, None, 'no'

matches = []
matched_kag_rows = []
for i, row in poly.iterrows():
    k, a_is_red, reason = match_one(row)
    if k is None: continue
    matches.append({
        'date': k['date'], 'a_is_red': a_is_red,
        'poly_a': row['fighter_a'], 'poly_b': row['fighter_b'],
        'closing_price_a': row['closing_price_a'],
        'closing_price_b': row['closing_price_b'],
        'poly_winner': row['winner'],
        'kag_R': k['R_fighter'], 'kag_B': k['B_fighter'],
        'kag_winner': k['Winner'],
        'match_reason': reason,
    })
    matched_kag_rows.append(k.name)

m_df = pd.DataFrame(matches)
matched_fights = fights.loc[matched_kag_rows].reset_index(drop=True)

# Reason breakdown
print(f"Matched: {len(m_df)}/{len(poly)} ({len(m_df)/len(poly)*100:.1f}%)")
print(f"Date range: {m_df['date'].min().date()} → {m_df['date'].max().date()}")
print("By pass:")
for r, n in m_df['match_reason'].value_counts().items():
    print(f"  {r:12s}: {n}")


## Compute model predictions on matched fights

In [ ]:
payload = joblib.load(ROOT / 'artifacts/models/v3_catboost_full2000_trainval.joblib')
X, _, _, _ = prepare(matched_fights, augment_symmetry=False, one_hot=False)
X = X.reindex(columns=payload['columns'], fill_value=None)
for c in payload.get('cat_features', []):
    X[c] = X[c].fillna('__missing__').astype(str)
p_red = payload['model'].predict_proba(X)[:, 1]
y_red = (matched_fights['Winner'].to_numpy() == 'Red').astype(int)

# Convert Polymarket probabilities → American odds in Red/Blue orientation.
def prob_to_american(p):
    p = float(p); dec = 1.0 / p
    return (dec - 1.0) * 100.0 if dec >= 2.0 else -100.0 / (dec - 1.0)

R_poly_odds, B_poly_odds = [], []
for r in matches:
    pa, pb = r['closing_price_a'], r['closing_price_b']
    if r['a_is_red']:
        R_poly_odds.append(prob_to_american(pa)); B_poly_odds.append(prob_to_american(pb))
    else:
        R_poly_odds.append(prob_to_american(pb)); B_poly_odds.append(prob_to_american(pa))
R_poly_odds = pd.Series(R_poly_odds)
B_poly_odds = pd.Series(B_poly_odds)

print(f'model pred mean: {p_red.mean():.3f}, range [{p_red.min():.3f}, {p_red.max():.3f}]')
print(f'y_red base rate: {y_red.mean():.3f}')

metrics = evaluate(y_red, p_red, label='polymarket_subset')
print(f"log_loss={metrics['log_loss']:.4f}  brier={metrics['brier']:.4f}  ece={metrics['ece']:.4f}  acc={metrics['accuracy_argmax']:.3f}")

## Orientation-symmetrized + sharpened predictions (live pipeline as of 2026-06-11)

The deployed models are not corner-symmetric (training augmentation didn't sign-flip
dif columns), so single-orientation predictions depend on arbitrary corner ordering.
The live pipeline now (a) averages each prediction over both corner orderings and
(b) sharpens the result with logit temperature T=1.25 (validated cross-window).
This section re-scores the same matched fights with both variants so the Polymarket
evaluation reflects what the live system actually serves.

In [ ]:
from ufc_pred.features.static_v1 import _swap_red_blue

mirrored = _swap_red_blue(matched_fights)
Xm, _, _, _ = prepare(mirrored, augment_symmetry=False, one_hot=False)
Xm = Xm.reindex(columns=payload['columns'], fill_value=None)
for c in payload.get('cat_features', []):
    Xm[c] = Xm[c].fillna('__missing__').astype(str)
p_mir = payload['model'].predict_proba(Xm)[:, 1]
p_sym = 0.5 * (p_red + 1.0 - p_mir)

def sharpen(p, t=1.25):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    z = np.log(p / (1 - p)) * t
    return 1.0 / (1.0 + np.exp(-z))

p_live = sharpen(p_sym, 1.25)

gap = p_red - (1.0 - p_mir)
print(f'orientation gap |p_fwd - (1-p_rev)|: mean={np.abs(gap).mean():.4f}  '
      f'median={np.median(np.abs(gap)):.4f}  max={np.abs(gap).max():.4f}')
for name, pp in (('original', p_red), ('symmetric', p_sym), ('symmetric+T1.25 (live)', p_live)):
    m = evaluate(y_red, pp, label=name)
    print(f"{name:24s} log_loss={m['log_loss']:.4f}  brier={m['brier']:.4f}  "
          f"ece={m['ece']:.4f}  acc={m['accuracy_argmax']:.3f}")

sym_rows = []
for thr in (0.03, 0.05):
    for name, pp in (('original', p_red), ('symmetric', p_sym), ('sym+T1.25 (live)', p_live)):
        r = evaluate_bets(pp, y_red, R_poly_odds, B_poly_odds,
                          edge_threshold=thr, fee_rate=0.02, use_no_vig=False)
        sym_rows.append({'edge_thr': f'{int(thr*100)}%', 'preds': name, 'n_bets': r.n_bets,
                         'roi_pct': r.roi_pct, 'hit_rate': r.hit_rate,
                         'ci95_lo': r.ci95_roi_pct[0], 'ci95_hi': r.ci95_roi_pct[1]})
pd.DataFrame(sym_rows)


## Flat-stake ROI on Polymarket prices — across fee scenarios and edge thresholds

Polymarket prices are already no-vig (probabilities sum to 1), so `use_no_vig=False`.

Three fee scenarios:
- **0% fee** — idealized; no trading costs.
- **2% fee** — realistic Polymarket cost (their fee structure is variable but ~2% on winnings is a good central estimate).
- **7% fee** — Kalshi's structure (for comparison with notebook 04's numbers).

**Watch the CI95 lower bound** — that's the credibility signal.

In [ ]:
rows = []
for fee in [0.0, 0.02, 0.07]:
    for thr in [0.03, 0.05, 0.10]:
        r = evaluate_bets(
            p_red, y_red, R_poly_odds, B_poly_odds,
            edge_threshold=thr, fee_rate=fee, use_no_vig=False,
        )
        rows.append({
            'fee_pct': f'{int(fee*100)}%',
            'edge_thr': f'{int(thr*100)}%',
            'n_bets': r.n_bets,
            'roi_pct': r.roi_pct,
            'ci95_low': r.ci95_roi_pct[0],
            'ci95_high': r.ci95_roi_pct[1],
            'hit_rate': r.hit_rate,
            'mean_ev_pct': r.mean_ev_pct,
        })
flat_df = pd.DataFrame(rows)
flat_df.style.format({
    'roi_pct': '{:+.3f}', 'ci95_low': '{:+.3f}', 'ci95_high': '{:+.3f}',
    'hit_rate': '{:.3f}', 'mean_ev_pct': '{:+.2f}',
})

## Side-by-side: Polymarket vs BestFightOdds-Kalshi (test eval)

Same model, same time period — different market pricing assumption.

In [ ]:
# Re-run BestFightOdds-Kalshi-like on the SAME 585 matched fights for apples-to-apples.
r_bfo = evaluate_bets(
    p_red, y_red, matched_fights['R_odds'], matched_fights['B_odds'],
    edge_threshold=0.05, fee_rate=0.07, use_no_vig=True,
)
r_poly = evaluate_bets(
    p_red, y_red, R_poly_odds, B_poly_odds,
    edge_threshold=0.05, fee_rate=0.02, use_no_vig=False,
)

compare = pd.DataFrame([{
    'market': 'BestFightOdds Kalshi-like (no-vig + 7% fee)',
    'n_bets': r_bfo.n_bets, 'roi_pct': r_bfo.roi_pct,
    'ci95_low': r_bfo.ci95_roi_pct[0], 'ci95_high': r_bfo.ci95_roi_pct[1],
    'hit_rate': r_bfo.hit_rate,
    'mean_ev_pct': r_bfo.mean_ev_pct,
}, {
    'market': 'Polymarket (real prices + 2% fee)',
    'n_bets': r_poly.n_bets, 'roi_pct': r_poly.roi_pct,
    'ci95_low': r_poly.ci95_roi_pct[0], 'ci95_high': r_poly.ci95_roi_pct[1],
    'hit_rate': r_poly.hit_rate,
    'mean_ev_pct': r_poly.mean_ev_pct,
}])
compare.style.format({
    'roi_pct': '{:+.3f}', 'ci95_low': '{:+.3f}', 'ci95_high': '{:+.3f}',
    'hit_rate': '{:.3f}', 'mean_ev_pct': '{:+.2f}',
})

## Kelly bankroll grid on Polymarket prices (2% fee)

Same 4×5 grid as notebooks 03 and 04, but priced against Polymarket. This is the realistic deployment expectation.

In [ ]:
FRACTIONS = [0.10, 0.25, 0.50, 1.00]
CAPS = [0.01, 0.02, 0.05, 0.10, 1.00]
FEE_RATE = 0.02   # Realistic Polymarket fee
EDGE_THRESHOLD = 0.03

grid = {}
for f in FRACTIONS:
    for c in CAPS:
        grid[(f, c)] = evaluate_bets_kelly(
            p_red, y_red, R_poly_odds, B_poly_odds,
            edge_threshold=EDGE_THRESHOLD, fee_rate=FEE_RATE, use_no_vig=False,
            kelly_fraction=f, max_bet_fraction=c, starting_bankroll=1.0,
        )

def grid_to_df(key):
    data = {('no cap' if c >= 1 else f'{int(c*100)}%'): [grid[(f, c)][key] for f in FRACTIONS] for c in CAPS}
    df = pd.DataFrame(data, index=[f'{int(f*100)}%-K' for f in FRACTIONS])
    df.index.name = 'Kelly fraction'; df.columns.name = 'per-bet cap'
    return df

bk = grid_to_df('final_bankroll')
dd = grid_to_df('max_drawdown_pct')

print('FINAL BANKROLL ($1 → ?) on Polymarket prices, 2% fee, edge ≥ 3%')
display(bk.style.format('${:,.2f}').background_gradient(cmap='RdYlGn', vmin=0.5, vmax=50, axis=None))

print('\nMAX DRAWDOWN (%)')
display(dd.style.format('{:.1f}%').background_gradient(cmap='RdYlGn_r', vmin=0, vmax=100, axis=None))

## Deploy-config trajectories on Polymarket

Same three accounts as DEPLOY.md, but priced against Polymarket. Starting bankroll $300 each.

In [ ]:
ACCOUNTS = [
    ('A', 'v3_real',  0.10, 0.10, '#1a9641'),
    ('B', 'v3_real',  0.25, 1.00, '#1f78b4'),
]
# Account C would use the corrupted model — load and predict.
payload_c = joblib.load(ROOT / 'artifacts/models/v3_full2000_no_skill_corrupted_trainval.joblib')
Xc = X.copy()  # reuse the prepared X
p_corrupt = payload_c['model'].predict_proba(Xc)[:, 1]

START = 300.0

def simulate(p, kelly, cap):
    return evaluate_bets_kelly(
        p, y_red, R_poly_odds, B_poly_odds,
        edge_threshold=EDGE_THRESHOLD, fee_rate=FEE_RATE, use_no_vig=False,
        kelly_fraction=kelly, max_bet_fraction=cap, starting_bankroll=START,
    )

sims = {
    'A': simulate(p_red, 0.10, 0.10),
    'B': simulate(p_red, 0.25, 1.00),
    'C': simulate(p_corrupt, 0.25, 1.00),
}
for tag, sim in sims.items():
    print(f'Account {tag}: ${START:.0f} → ${sim["final_bankroll"]:,.2f}  '
          f'max DD {sim["max_drawdown_pct"]:.1f}%  n_bets={sim["n_bets"]}')

In [ ]:
colors = {'A': '#1a9641', 'B': '#1f78b4', 'C': '#d7191c'}
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=False)

for ax, scale in zip(axes, ['linear', 'log']):
    for tag, sim in sims.items():
        ax.plot(sim['trajectory'], color=colors[tag], linewidth=2.0,
                label=f'Account {tag}: ${START:.0f} → ${sim["final_bankroll"]:,.2f}')
    ax.axhline(START, color='black', alpha=0.4, linewidth=0.8)
    ax.set_yscale(scale)
    ax.set_xlabel('Bet # (chronological through Polymarket-matched test fights)')
    ax.set_ylabel(f'Bankroll ($) — {scale}')
    ax.set_title(f'Deploy accounts on Polymarket prices ({scale} scale)')
    ax.legend(loc='upper left' if scale == 'linear' else 'lower right')
    ax.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

## Side-by-side: deploy-config bankrolls (BestFightOdds-Kalshi vs Polymarket)

What did the test eval (notebook 04) project vs what Polymarket pricing actually delivers on the matched subset?

In [ ]:
# Same accounts but on BestFightOdds-Kalshi-like, restricted to the 585 matched fights.
def simulate_bfo(p, kelly, cap):
    return evaluate_bets_kelly(
        p, y_red, matched_fights['R_odds'], matched_fights['B_odds'],
        edge_threshold=0.03, fee_rate=0.07, use_no_vig=True,
        kelly_fraction=kelly, max_bet_fraction=cap, starting_bankroll=START,
    )
sims_bfo = {
    'A': simulate_bfo(p_red, 0.10, 0.10),
    'B': simulate_bfo(p_red, 0.25, 1.00),
    'C': simulate_bfo(p_corrupt, 0.25, 1.00),
}

tbl = pd.DataFrame([{
    'account': tag,
    'config': ('10%-K + 10% cap' if tag == 'A' else '¼-K + no cap'),
    'BFO Kalshi-like ($300 → ?)': sims_bfo[tag]['final_bankroll'],
    'Polymarket 2% ($300 → ?)':   sims[tag]['final_bankroll'],
    'ratio': sims_bfo[tag]['final_bankroll'] / sims[tag]['final_bankroll'],
} for tag in ['A','B','C']])
tbl.style.format({
    'BFO Kalshi-like ($300 → ?)': '${:,.2f}',
    'Polymarket 2% ($300 → ?)':   '${:,.2f}',
    'ratio': '{:.1f}×',
})

## Per-fight-night Polymarket P&L for each account

Same per-night view as notebook 04, but priced against Polymarket.

In [ ]:
# Build per-bet log per account using simulate_with_logging-style mechanics.
def simulate_log(p, kelly, cap):
    R, B = R_poly_odds.to_numpy(), B_poly_odds.to_numpy()
    dec_R = 1.0 / np.where(np.asarray([r['a_is_red'] for r in matches]),
                            [r['closing_price_a'] for r in matches],
                            [r['closing_price_b'] for r in matches])
    dec_B = 1.0 / np.where(np.asarray([r['a_is_red'] for r in matches]),
                            [r['closing_price_b'] for r in matches],
                            [r['closing_price_a'] for r in matches])
    eff_R = 1.0 + (1.0 - FEE_RATE) * (dec_R - 1.0)
    eff_B = 1.0 + (1.0 - FEE_RATE) * (dec_B - 1.0)
    p_R = np.asarray(p); p_B = 1.0 - p_R
    ev_R = p_R * eff_R - 1.0; ev_B = p_B * eff_B - 1.0
    bet_red = ev_R >= ev_B
    chosen_ev = np.where(bet_red, ev_R, ev_B)
    chosen_dec = np.where(bet_red, eff_R, eff_B)
    chosen_p = np.where(bet_red, p_R, p_B)
    bets_mask = chosen_ev > EDGE_THRESHOLD
    won_full = np.where(bet_red, y_red == 1, y_red == 0)
    bankroll = START; rows = []
    for i in range(len(p_R)):
        if not bets_mask[i]: continue
        b = chosen_dec[i] - 1.0
        pi = chosen_p[i]; qi = 1.0 - pi
        fk = (b*pi - qi) / b
        if fk <= 0: continue
        stake_frac = min(kelly * fk, cap)
        stake = bankroll * stake_frac
        if won_full[i]: bankroll += stake * (chosen_dec[i] - 1.0)
        else: bankroll -= stake
        rows.append({'date': matches[i]['date'], 'won': int(won_full[i]),
                     'stake': stake, 'bankroll_after': bankroll})
    return pd.DataFrame(rows)

logs = {
    'A': simulate_log(p_red, 0.10, 0.10),
    'B': simulate_log(p_red, 0.25, 1.00),
    'C': simulate_log(p_corrupt, 0.25, 1.00),
}

def per_night(log):
    g = log.groupby('date', sort=True)
    out = g.agg(n_bets=('won','size'), n_wins=('won','sum'),
                bankroll_end=('bankroll_after','last')).reset_index()
    prev = np.concatenate([[START], out['bankroll_end'].iloc[:-1].to_numpy()])
    out['pnl'] = out['bankroll_end'] - prev
    return out

nights = {tag: per_night(log) for tag, log in logs.items()}
for tag, n in nights.items():
    print(f'Account {tag}: {len(n)} fight nights, ${START:.0f} → ${n["bankroll_end"].iloc[-1]:,.2f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
for ax, scale in zip(axes, ['linear', 'log']):
    for tag, n in nights.items():
        ax.plot(n['date'], n['bankroll_end'], color=colors[tag], linewidth=1.8,
                label=f'Account {tag}: ${START:.0f} → ${n["bankroll_end"].iloc[-1]:,.2f}')
    ax.axhline(START, color='black', alpha=0.4, linewidth=0.8)
    ax.set_yscale(scale)
    ax.set_ylabel(f'Bankroll ($) — {scale}')
    ax.set_title(f'Per-fight-night bankroll on Polymarket prices ({scale})')
    ax.legend(loc='upper left' if scale == 'linear' else 'lower right')
    ax.grid(alpha=0.3, which='both')
axes[1].set_xlabel('Date (fight night)')
plt.tight_layout(); plt.show()

In [ ]:
# Top 5 best and worst nights per account, Polymarket pricing
out = []
for tag, n in nights.items():
    best = n.nlargest(5, 'pnl').assign(rank='best', account=tag)
    worst = n.nsmallest(5, 'pnl').assign(rank='worst', account=tag)
    out.append(pd.concat([best, worst]))
moves = pd.concat(out, ignore_index=True)[['account','rank','date','n_bets','n_wins','pnl','bankroll_end']]
moves['pnl'] = moves['pnl'].round(2)
moves['bankroll_end'] = moves['bankroll_end'].round(2)
moves

## Calibration on Polymarket-matched bets

Same overconfidence finding as in METRICS_AUDIT.md, but on the Polymarket-restricted subset.

In [ ]:
r_poly_5 = evaluate_bets(
    p_red, y_red, R_poly_odds, B_poly_odds,
    edge_threshold=0.05, fee_rate=0.02, use_no_vig=False,
)
bd = r_poly_5.bets_df
bins = [(0.5,0.6),(0.6,0.7),(0.7,0.8),(0.8,0.9),(0.9,1.0)]
rows = []
for lo, hi in bins:
    m = (bd['model_prob_chosen'] >= lo) & (bd['model_prob_chosen'] < hi)
    if m.sum() == 0: continue
    rows.append({
        'bin': f'[{lo:.1f}, {hi:.1f})',
        'n_bets': int(m.sum()),
        'avg_claimed_p': bd.loc[m, 'model_prob_chosen'].mean(),
        'realized_hit': bd.loc[m, 'won'].mean(),
        'gap_pp': (bd.loc[m, 'won'].mean() - bd.loc[m, 'model_prob_chosen'].mean()) * 100,
    })
calib_df = pd.DataFrame(rows)
calib_df.style.format({
    'avg_claimed_p': '{:.3f}', 'realized_hit': '{:.3f}', 'gap_pp': '{:+.1f}',
})

## Walkthrough — the first 10 bets in detail

To make the Kelly math concrete, here is the per-bet calculation for the first 10 fights that **Account B** (¼-Kelly, no cap, real model, starting bankroll $300) actually placed a bet on.

Each row shows every input and output the simulation uses for that bet:

| Column | Meaning |
|---|---|
| `bet_idx` | Sequence number (chronological through matched fights) |
| `date` | Fight date |
| `chosen` | Fighter we bet on |
| `model_p` | Model's claimed P(chosen side wins) |
| `poly_p`  | Polymarket's closing P(chosen side wins) |
| `dec`     | Decimal odds = 1 / poly_p |
| `eff_dec` | Post-fee decimal = 1 + (1 − fee) × (dec − 1), with fee = 2% |
| `EV%`     | Model EV per $1 = model_p × eff_dec − 1 |
| `full_K`  | Full Kelly fraction = (b·p − q) / b, where b = eff_dec − 1, p = model_p, q = 1−p |
| `¼-K %`   | Stake fraction = min(0.25 × full_K, 1.0) = bankroll fraction to stake |
| `stake`   | $ to stake = bankroll × ¼-K % |
| `won`     | Did the chosen side win? |
| `realized`| PnL on the bet = stake × (eff_dec − 1) if win, else −stake |
| `bankroll`| New bankroll after this bet |

In [ ]:
# Re-implement the Kelly simulation with full per-bet logging for Account B.
START = 300.0
FEE = 0.02
KELLY_FRAC = 0.25
CAP = 1.0
EDGE_THR = 0.03

# Build per-fight arrays in Kaggle Red/Blue orientation.
pa = np.array([m['closing_price_a'] for m in matches])
pb = np.array([m['closing_price_b'] for m in matches])
a_is_red = np.array([m['a_is_red'] for m in matches])

# Convert Polymarket probability → effective post-fee decimal payout, per side.
dec_R = np.where(a_is_red, 1.0/pa, 1.0/pb)
dec_B = np.where(a_is_red, 1.0/pb, 1.0/pa)
eff_R = 1.0 + (1.0 - FEE) * (dec_R - 1.0)
eff_B = 1.0 + (1.0 - FEE) * (dec_B - 1.0)

p_R = p_red
p_B = 1.0 - p_R
ev_R = p_R * eff_R - 1.0
ev_B = p_B * eff_B - 1.0
bet_red = ev_R >= ev_B
chosen_ev = np.where(bet_red, ev_R, ev_B)
chosen_dec_eff = np.where(bet_red, eff_R, eff_B)
chosen_p_model = np.where(bet_red, p_R, p_B)
chosen_p_poly  = np.where(bet_red,
                          np.where(a_is_red, pa, pb),
                          np.where(a_is_red, pb, pa))
chosen_name = np.where(bet_red,
                       matched_fights['R_fighter'].to_numpy(),
                       matched_fights['B_fighter'].to_numpy())
won_full = np.where(bet_red,
                    matched_fights['Winner'].to_numpy() == 'Red',
                    matched_fights['Winner'].to_numpy() == 'Blue').astype(int)
bets_mask = chosen_ev > EDGE_THR

# Walk forward
bankroll = START
log = []
bet_idx = 0
for i in range(len(matches)):
    if not bets_mask[i]:
        continue
    b = chosen_dec_eff[i] - 1.0
    p = chosen_p_model[i]
    q = 1.0 - p
    full_kelly = (b*p - q) / b
    if full_kelly <= 0:
        continue
    stake_frac = min(KELLY_FRAC * full_kelly, CAP)
    stake = bankroll * stake_frac
    if won_full[i]:
        realized = stake * (chosen_dec_eff[i] - 1.0)
    else:
        realized = -stake
    bankroll_after = bankroll + realized
    log.append({
        'bet_idx': bet_idx,
        'date': matched_fights['date'].iloc[i].date(),
        'chosen': chosen_name[i],
        'model_p': p,
        'poly_p':  chosen_p_poly[i],
        'dec':     1.0 / chosen_p_poly[i],
        'eff_dec': chosen_dec_eff[i],
        'EV%':     chosen_ev[i] * 100,
        'full_K':  full_kelly,
        'qK_pct':  stake_frac * 100,
        'stake':   stake,
        'won':     int(won_full[i]),
        'realized': realized,
        'bankroll': bankroll_after,
    })
    bankroll = bankroll_after
    bet_idx += 1

log_df = pd.DataFrame(log)
print(f'Total bets placed: {len(log_df)}')
print(f'Bankroll at end of first 10 bets: ${log_df["bankroll"].iloc[9]:,.2f}')
print(f'Final bankroll: ${log_df["bankroll"].iloc[-1]:,.2f}')

# Display the first 10 bets with formatted columns
first10 = log_df.head(10).copy()
first10.style.format({
    'model_p': '{:.3f}', 'poly_p': '{:.3f}', 'dec': '{:.3f}', 'eff_dec': '{:.3f}',
    'EV%': '{:+.2f}%', 'full_K': '{:.3f}', 'qK_pct': '{:.2f}%',
    'stake': '${:,.2f}', 'realized': '${:+,.2f}', 'bankroll': '${:,.2f}',
})


### Reading the first 10 bets

A few things to notice in the table above:

1. **The stake fractions are mostly far below 25%.** Most bets stake 1–8% of bankroll. Full Kelly for an edge like model_p=0.55 on a fair-50/50 market is small (~10%), so ¼-K is 2.5%. Only the high-confidence model picks (model_p > 0.85 on a fair line) stake double-digit %.

2. **Bankroll moves in line with stake × outcome.** A $30 bet at +200 odds (decimal 3.0, eff_dec 2.96) that wins adds ~$58. The same bet that loses subtracts $30. This is what compounding actually looks like — small steps, occasional big steps when the model is confident.

3. **Wins early are very compounding-significant.** A win in the first 10 bets matters more than a win in the last 10 because the win amount is a fraction of starting bankroll (small), but the *survival of bankroll* across early bets is what enables all later compounding. An early loss is recoverable; an early sequence of 3-4 max-confidence losses is what kills no-cap Kelly trajectories (see the verification cells in notebook 04).

4. **The chosen side is whichever has higher EV after fees**, not always the model's higher-probability side. On a heavily-favored Red where Polymarket also has Red at high price, the EV might be higher on Blue (the model thinks Blue is more likely than the market says).

5. **EV% is what the model THINKS the bet is worth.** As METRICS_AUDIT.md noted, these EV claims are systematically inflated (the model is overconfident). Realized per-bet returns are much smaller on average, but still positive in aggregate.

## Walkthrough — Account C (corrupted model) first 10 bets

Same configuration as Account B (¼-Kelly, no cap, $300 starting, 2% Polymarket fee, edge ≥ 3%) but using the **corrupted-skill model** (`v3_full2000_no_skill_corrupted_trainval`), which forces NaN on the `skill_diff_mean` and `skill_diff_std` columns at inference.

This makes the model's probabilities **less confident on average** (no Bayesian skill signal), so Kelly stakes are smaller per bet. The interesting question: same fights, smaller stakes — does that help or hurt?

In [ ]:
# Re-implement Kelly with full per-bet logging for Account C (corrupted model).
START_C = 300.0  # same starting capital
FEE_C = 0.02
KELLY_FRAC_C = 0.25
CAP_C = 1.0
EDGE_THR_C = 0.03

# Polymarket prices in Kaggle Red/Blue orientation (same as Account B).
pa_c = np.array([m['closing_price_a'] for m in matches])
pb_c = np.array([m['closing_price_b'] for m in matches])
a_is_red_c = np.array([m['a_is_red'] for m in matches])

dec_R_c = np.where(a_is_red_c, 1.0/pa_c, 1.0/pb_c)
dec_B_c = np.where(a_is_red_c, 1.0/pb_c, 1.0/pa_c)
eff_R_c = 1.0 + (1.0 - FEE_C) * (dec_R_c - 1.0)
eff_B_c = 1.0 + (1.0 - FEE_C) * (dec_B_c - 1.0)

# Use CORRUPTED model probabilities here (not p_red)
p_R_c = p_corrupt
p_B_c = 1.0 - p_R_c
ev_R_c = p_R_c * eff_R_c - 1.0
ev_B_c = p_B_c * eff_B_c - 1.0
bet_red_c = ev_R_c >= ev_B_c
chosen_ev_c = np.where(bet_red_c, ev_R_c, ev_B_c)
chosen_dec_eff_c = np.where(bet_red_c, eff_R_c, eff_B_c)
chosen_p_model_c = np.where(bet_red_c, p_R_c, p_B_c)
chosen_p_poly_c  = np.where(bet_red_c,
                            np.where(a_is_red_c, pa_c, pb_c),
                            np.where(a_is_red_c, pb_c, pa_c))
chosen_name_c = np.where(bet_red_c,
                         matched_fights['R_fighter'].to_numpy(),
                         matched_fights['B_fighter'].to_numpy())
won_full_c = np.where(bet_red_c,
                      matched_fights['Winner'].to_numpy() == 'Red',
                      matched_fights['Winner'].to_numpy() == 'Blue').astype(int)
bets_mask_c = chosen_ev_c > EDGE_THR_C

bankroll_c = START_C
log_c = []
bet_idx_c = 0
for i in range(len(matches)):
    if not bets_mask_c[i]:
        continue
    b = chosen_dec_eff_c[i] - 1.0
    p = chosen_p_model_c[i]
    q = 1.0 - p
    full_kelly = (b*p - q) / b
    if full_kelly <= 0:
        continue
    stake_frac = min(KELLY_FRAC_C * full_kelly, CAP_C)
    stake = bankroll_c * stake_frac
    if won_full_c[i]:
        realized = stake * (chosen_dec_eff_c[i] - 1.0)
    else:
        realized = -stake
    bankroll_c += realized
    log_c.append({
        'bet_idx': bet_idx_c,
        'date': matched_fights['date'].iloc[i].date(),
        'chosen': chosen_name_c[i],
        'model_p': p,
        'poly_p':  chosen_p_poly_c[i],
        'dec':     1.0 / chosen_p_poly_c[i],
        'eff_dec': chosen_dec_eff_c[i],
        'EV%':     chosen_ev_c[i] * 100,
        'full_K':  full_kelly,
        'qK_pct':  stake_frac * 100,
        'stake':   stake,
        'won':     int(won_full_c[i]),
        'realized': realized,
        'bankroll': bankroll_c,
    })
    bet_idx_c += 1

log_c_df = pd.DataFrame(log_c)
print(f'Total bets placed (Account C): {len(log_c_df)}')
print(f'Bankroll at end of first 10 bets: ${log_c_df["bankroll"].iloc[9]:,.2f}')
print(f'Final bankroll: ${log_c_df["bankroll"].iloc[-1]:,.2f}')

first10_c = log_c_df.head(10).copy()
first10_c.style.format({
    'model_p': '{:.3f}', 'poly_p': '{:.3f}', 'dec': '{:.3f}', 'eff_dec': '{:.3f}',
    'EV%': '{:+.2f}%', 'full_K': '{:.3f}', 'qK_pct': '{:.2f}%',
    'stake': '${:,.2f}', 'realized': '${:+,.2f}', 'bankroll': '${:,.2f}',
})


In [ ]:
# Side-by-side: same first 10 bets, Account B (real) vs Account C (corrupted)
def fmt_money(x): return f'${x:,.2f}'

side_by_side = pd.DataFrame({
    'date':    log_df['date'].head(10).astype(str),
    'chosen_B': log_df['chosen'].head(10).str.slice(0, 18),
    'B_model_p': log_df['model_p'].head(10).round(3),
    'B_qK%':   log_df['qK_pct'].head(10).round(2),
    'B_stake': log_df['stake'].head(10).apply(fmt_money),
    'B_bank':  log_df['bankroll'].head(10).apply(fmt_money),
    'chosen_C': log_c_df['chosen'].head(10).str.slice(0, 18),
    'C_model_p': log_c_df['model_p'].head(10).round(3),
    'C_qK%':   log_c_df['qK_pct'].head(10).round(2),
    'C_stake': log_c_df['stake'].head(10).apply(fmt_money),
    'C_bank':  log_c_df['bankroll'].head(10).apply(fmt_money),
})
side_by_side


### What changed for Account C

Comparing the two walkthroughs (Account B real model vs Account C corrupted model) on the same 10 fights:

1. **Corrupted model probabilities differ from real, but NOT uniformly toward 0.5.** Removing the Bayesian skill feature doesn't just dampen predictions — the model re-weights remaining features. On some fights the corrupted model is *less* confident (Hill 0.295 → 0.283, Brundage 0.216 → 0.197, Prochazka 0.741 → 0.701) but on others it's *more* confident (Sterling 0.641 → 0.732, Zhang Weili 0.877 → 0.909). This reshuffling is why corruption isn't equivalent to calibration shrinkage — it's a different model.

2. **The two models even disagree on which side to bet.** Bet #6 in this sample is the Justin Gaethje vs Max Holloway fight: the real model favored Holloway (won +$16.57), the corrupted model favored Gaethje (lost −$42.72). Same fight, opposite sides, ~$59 swing on a single bet between the two accounts.

3. **Bankroll after 10 bets diverges.** Account B: $300 → $331.15 (+10.4%). Account C: $300 → $271.63 (−9.5%). The Gaethje/Holloway disagreement plus a few smaller differences leave C ~$60 behind B at this point in the simulation.

4. **But over the full 615 bets, Account C ends much higher.** The first-10-bets sample is unrepresentative. By the end of the matched test window, Account C ($7.5M from $300) substantially exceeds Account B ($484k from $300). This is the same pattern observed on val and the test eval (notebook 04): at no-cap Kelly, the corrupted variant's accidentally-different bet sizing avoids catastrophic single-bet wipeouts the real model is susceptible to. The mechanism is luck-shaped — the corrupted model happens to dodge specific high-confidence losses that hit the real model — and is documented in STATUS.md as the "ensemble paradox / corrupted variant" finding.

5. **At conservative sizing the real model still wins.** From notebook 04 / DEPLOY.md context: at ¼-K + 2% cap (PLAN.md default), the real model outperforms corrupted on both val and test. The corrupted advantage is tied to no-cap regimes where variance reduction helps more than signal density. The first 10 bets here are a microcosm of that variance — sometimes C wins by dodging losses, sometimes it loses by missing wins.

## Takeaways

1. **The model has a positive edge on Polymarket prices that's essentially the same as the BFO-Kalshi-like simulation in notebook 04.** At 2% fee, edge ≥ 5%, on 666 matched fights: **+10.85% ROI**, CI95 = **(+0.87%, +21.79%)**. The full-test BFO-Kalshi-like number was +10.88% with CI95 (+2.75%, +19.39%) on 884 bets. Per-bet geometric growth on Polymarket (1.0121) is actually slightly above BFO-Kalshi-like (1.0093).

2. **CI95 lower bound is above zero on Polymarket data.** +0.87% lower bound means the demonstrated edge survives bootstrap reweighting of which bets land in the sample. This is the most important credibility signal in the analysis. It was negative in the initial 585-match version of this notebook; the improvement came from better fight matching, not from changing the model or the market.

3. **Polymarket vs BestFightOdds-Kalshi-like — same per-bet edge, smaller bankroll only because of bet count.** Compounding 615 Polymarket bets at 1.0121 per bet gives $1 → $1,613. Extrapolating to 884 bets at the same rate would give ~$25,000. The "$45k vs $1M" framing in earlier drafts of this notebook was an artifact of comparing different-length compounding windows.

4. **Hit rate.** 0.521 (Polymarket bets) vs 0.544 (BFO-Kalshi full test). Slightly lower but accompanied by higher decimal odds (Polymarket prices clear at slightly more favorable lines for the model's contrarian bets) — net effect is a comparable ROI.

5. **Bankroll growth on Polymarket pricing for $300 starting capital:**
   - Account A (10%-K + 10% cap, real): $300 → ~$5,800 (val 666 fights, 2.2 years)
   - Account B (¼-K + no cap, real):    $300 → ~$45,000–$485,000 depending on bet count compounding window
   - Account C (¼-K + no cap, corrupted): scaled correspondingly higher

6. **Calibration overconfidence is the same on Polymarket-matched bets.** The model continues to overstate its probabilities; the ¼-Kelly factor remains the implicit calibration mechanism. This is a model property, not a market property.

7. **265 unmatched Polymarket fights (28.5%).** Of these, ~159 are post-test (Kaggle ends 2026-03-28) and ~106 are within-window fights that are not in Kaggle at all — typically cancelled / replaced matchups (e.g. Polymarket listed Makhachev vs Tsarukyan at UFC 311, but Tsarukyan pulled out and the actual fight was Makhachev vs Moicano). These are data-quality limits, not matching-algorithm limits.

8. **Deployment implication.** Polymarket is a credible deployment market. The DEPLOY.md projections from the test eval are roughly correct when re-anchored to Polymarket pricing. The CI lower bound being just barely positive (+0.87%) is still a yellow flag — the edge could plausibly be small. The risk profile of no-cap Kelly remains as described in DEPLOY.md.